In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import plotly.io as pio
pio.renderers.default = 'jupyterlab' # or 'notebook_connected'

In [3]:
from importlib import reload

In [18]:
from math import *

In [4]:
import testObjects

In [5]:
reload(testObjects)

<module 'testObjects' from '/Users/espillar/Desktop/vibevolts/testObjects.py'>

## "By Hand" flux and noise calculation
### Outline:
0. keep track of units the whole way in secondary lines
1. Choose V band
2. Get the solar brightness in magnitudes
3. Convert to photons/sec on a $1 m^2$ satellite
4. Assume face on, calculate the lambertian flux at earth or at that range
5. multiply by area of the telescope and any efficiency factors
6. Multiply by integration time
7. output this number
8. Find the background flux
9. convert to photons per second per meters per $\Omega$
10. multply by area, $\Omega$, efficiency, integration tiem
11. output this number
12. computer SNR by flux over $\sqrt{noise}$
13. output this number
14. Compare, when it works convert it to LaTeX and put in equations document

In [20]:
def amag(m):  return(10**(-0.4 * m))
def mag(f): return( -2.5 * log10(f))
print(amag(5))
print(mag(100))

0.01
-5.0


V band solar brightness is -26.74 accounding to gemini, radiometry_data has it as -26.78. Agreement. 

Gemini makes this as $5.1 \times 10^{20}$ photons per second per square meter.  Using the constants in my head I get 

In [22]:
print(amag(-26.74) * 866000 * 10000)

4.300489503759908e+20


Which I think is good  agreement.  Now lets get the lambertian flux at earth. Well, I can get a complete thing out of gemini- I know I'm being lazy, but this is expedient.  

In [23]:
import numpy as np

def calculate_satellite_magnitude(radius, distance, albedo, phase_angle_deg):
    """
    Calculates the apparent V-band magnitude of a spherical satellite.
    """
    m_sun = -26.74
    alpha_rad = np.radians(phase_angle_deg)
    
    # 1. Get the Lambertian Phase Function value
    # Note: We divide by the peak value at alpha=0 (2/(3*pi)*pi) 
    # to use it as a scaling factor between 0 and 1.
    phi = (1 / np.pi) * ((np.pi - alpha_rad) * np.cos(alpha_rad) + np.sin(alpha_rad))
    
    # 2. Calculate the brightness ratio
    # (Albedo * Area_ratio * Phase)
    # The (2/3) comes from the integration of a diffuse sphere's brightness
    brightness_ratio = albedo * (2/3) * (radius**2 / distance**2) * phi
    
    # 3. Convert to magnitude
    mag = m_sun - 2.5 * np.log10(brightness_ratio)
    
    return mag

# Example: A 2-meter radius satellite at 550km altitude (LEO) 
# at a 30-degree phase angle with an albedo of 0.5
sat_mag = calculate_satellite_magnitude(radius=2, distance=100000000, albedo=0.2, phase_angle_deg=0)

print(f"Satellite Apparent Magnitude: {sat_mag:.2f}")

Satellite Apparent Magnitude: 13.94


## Code Tests

In [6]:
fig = testObjects.demoFixed()

--- Running scandetector ---
sun, space, sky  4.529e+20, 2.360e+11, 6.499e+11
SignAl, noise, snr, integration time 

5.967e-15, 1.547e+05, 3.858e-20, 1.291e-01
5.967e-17, 1.547e+05, 3.858e-22, 1.291e-01
Output of scandetector: 0
